# MPE2 Simple Tag: Init + Visualization

This notebook initializes the **Simple Tag** environment from MPE2 and visualizes a rollout with random actions.

In [1]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import animation
from IPython.display import HTML

from mpe2 import simple_tag_v3

/vol/bitbucket/jhl323/fypenv/lib/python3.12/site-packages/pygame/pkgdata.py:25: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_stream, resource_exists


In [8]:
# Environment config
NUM_GOOD = 1
NUM_ADVERSARIES = 3
NUM_OBSTACLES = 2
MAX_CYCLES = 100

env = simple_tag_v3.env(
    num_good=NUM_GOOD,
    num_adversaries=NUM_ADVERSARIES,
    num_obstacles=NUM_OBSTACLES,
    max_cycles=MAX_CYCLES,
    continuous_actions=False,
    render_mode="rgb_array",
)

print("Agents:", env.possible_agents)
for agent in env.possible_agents:
    print(f"{agent}: action_space={env.action_space(agent)}, obs_space={env.observation_space(agent)}")

Agents: ['adversary_0', 'adversary_1', 'adversary_2', 'agent_0']
adversary_0: action_space=Discrete(5), obs_space=Box(-inf, inf, (16,), float32)
adversary_1: action_space=Discrete(5), obs_space=Box(-inf, inf, (16,), float32)
adversary_2: action_space=Discrete(5), obs_space=Box(-inf, inf, (16,), float32)
agent_0: action_space=Discrete(5), obs_space=Box(-inf, inf, (14,), float32)


In [9]:
def collect_rollout_frames(env, max_iter=600, seed=0):
    env.reset(seed=seed)
    frames = []

    for agent in env.agent_iter(max_iter=max_iter):
        obs, reward, terminated, truncated, info = env.last()

        if terminated or truncated:
            action = None
        else:
            action = env.action_space(agent).sample()

        env.step(action)
        frame = env.render()
        if frame is not None:
            frames.append(frame)

    env.close()
    return frames

frames = collect_rollout_frames(env, max_iter=600, seed=42)
print(f"Collected {len(frames)} frames")

Collected 404 frames


In [10]:
def animate_frames(frames, interval=60):
    fig, ax = plt.subplots(figsize=(6, 6))
    ax.axis("off")

    img = ax.imshow(frames[0])

    def _update(i):
        img.set_data(frames[i])
        return (img,)

    anim = animation.FuncAnimation(
        fig,
        _update,
        frames=len(frames),
        interval=interval,
        blit=True,
    )
    plt.close(fig)
    return HTML(anim.to_jshtml())

animate_frames(frames[:250], interval=50)

## Train DDQN SRQ Agents

Train one `DDqnSrqAgent` per MPE2 agent for 100 episodes, then plot per-agent episode rewards.

In [ ]:
import sys
from pathlib import Path
from collections import defaultdict
from tqdm.auto import trange

# Make local DQN module imports robust regardless of notebook launch directory.
candidate_dirs = [Path.cwd(), Path.cwd() / "discrete_action_space" / "DQN"]
for d in candidate_dirs:
    if (d / "DoubleDQN.py").exists():
        if str(d) not in sys.path:
            sys.path.insert(0, str(d))
        break

from DoubleDQN import DDqnSrqAgent

In [ ]:
EPISODES = 100
BATCH_SIZE = 64
TARGET_UPDATE_EVERY = 10
SEED = 123

train_env = simple_tag_v3.parallel_env(
    num_good=NUM_GOOD,
    num_adversaries=NUM_ADVERSARIES,
    num_obstacles=NUM_OBSTACLES,
    max_cycles=MAX_CYCLES,
    continuous_actions=False,
    render_mode=None,
)

# Initialize once to inspect spaces and build agents.
reset_out = train_env.reset(seed=SEED)
if isinstance(reset_out, tuple):
    obs_dict, _ = reset_out
else:
    obs_dict = reset_out

agent_names = list(train_env.possible_agents)
num_agents = len(agent_names)
obs_dims = {a: int(np.asarray(train_env.observation_space(a).shape).prod()) for a in agent_names}
num_actions = {a: train_env.action_space(a).n for a in agent_names}

agents = {}
for idx, agent_name in enumerate(agent_names):
    agents[agent_name] = DDqnSrqAgent(
        agent_id=idx,
        obs_dim=obs_dims[agent_name],
        num_agents=num_agents,
        num_actions=num_actions[agent_name],
        epsilon_explore=1.0,
        gamma=0.95,
        lr=1e-3,
        decay_rate=0.995,
        buffer_size=10000,
        use_gpu=True,
    )

reward_history = {a: [] for a in agent_names}

for ep in trange(EPISODES, desc="Training DDQN"):
    reset_out = train_env.reset(seed=SEED + ep)
    if isinstance(reset_out, tuple):
        obs_dict, _ = reset_out
    else:
        obs_dict = reset_out

    episode_rewards = defaultdict(float)

    while train_env.agents:
        active_agents = list(train_env.agents)

        actions = {a: agents[a].act(obs_dict[a]) for a in active_agents}

        next_obs, rewards, terminations, truncations, infos = train_env.step(actions)

        for a in active_agents:
            done = terminations.get(a, False) or truncations.get(a, False)
            next_state = next_obs.get(a, np.zeros(obs_dims[a], dtype=np.float32))
            r = rewards.get(a, 0.0)

            agents[a].update(
                state=obs_dict[a],
                actions=actions[a],
                rewards=r,
                next_state=next_state,
                done=done,
                batch_size=BATCH_SIZE,
            )
            episode_rewards[a] += r

        obs_dict = next_obs

    for a in agent_names:
        reward_history[a].append(float(episode_rewards[a]))

    for ag in agents.values():
        ag.decay_parameters()

    if (ep + 1) % TARGET_UPDATE_EVERY == 0:
        for ag in agents.values():
            ag.update_target_network()

train_env.close()
print("Training complete.")

In [ ]:
plt.figure(figsize=(10, 5))
for agent_name, rewards in reward_history.items():
    plt.plot(rewards, label=agent_name, alpha=0.85)

plt.xlabel("Episode")
plt.ylabel("Episode Reward")
plt.title("DDQN SRQ Training Reward Curve (Simple Tag)")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()